In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers

2025-11-02 06:21:34.662017: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
column_names = [
    "age", "workclass", "fnlwgt", "education", "education.num",
    "marital.status", "occupation", "relationship", "race", "sex",
    "capital.gain", "capital.loss", "hours.per.week",
    "native.country", "income"
]

train_path = "../data/adult/adult.data"  
test_path  = "../data/adult/adult.test"

df_train = pd.read_csv(
    train_path, 
    header=None, 
    names=column_names,   
    na_values="?"
)
df_test = pd.read_csv(
    test_path, 
    header=None, 
    names=column_names,
    na_values="?",
    skiprows=1  # Uncomment if your test file has a header or junk line
)

In [3]:
X_train = df_train.drop("income", axis=1)
y_train = df_train["income"]

X_test = df_test.drop("income", axis=1)
y_test = df_test["income"]

In [4]:
cat_cols = [
    "workclass", "education", "marital.status", "occupation",
    "relationship", "race", "sex", "native.country"
]

combined = pd.concat([X_train, X_test], keys=["train", "test"])

# One-hot encode
combined = pd.get_dummies(combined, columns=cat_cols)

# Separate back into X_train, X_test
X_train = combined.loc["train"]
X_test  = combined.loc["test"]

In [5]:
num_cols = ["age", "fnlwgt", "education.num", "capital.gain",
            "capital.loss", "hours.per.week"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train

/tmp/ipykernel_20991/3114243991.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
/tmp/ipykernel_20991/3114243991.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[num_cols] = scaler.transform(X_test[num_cols])


,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,workclass_ ?,workclass_ Federal-gov,workclass_ Local-gov,workclass_ Never-worked,...,native.country_ Portugal,native.country_ Puerto-Rico,native.country_ Scotland,native.country_ South,native.country_ Taiwan,native.country_ Thailand,native.country_ Trinadad&Tobago,native.country_ United-States,native.country_ Vietnam,native.country_ Yugoslavia
0,0.030671,-1.063611,1.134739,0.148453,-0.21666,-0.035429,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,0.837109,-1.008707,1.134739,-0.145920,-0.21666,-2.222153,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,-0.042642,0.245079,-0.420060,-0.145920,-0.21666,-0.035429,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,1.057047,0.425801,-1.197459,-0.145920,-0.21666,-0.035429,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,-0.775768,1.408176,1.134739,-0.145920,-0.21666,-0.035429,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,-0.849080,0.639741,0.746039,-0.145920,-0.21666,-0.197409,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
32557,0.103983,-0.335433,-0.420060,-0.145920,-0.21666,-0.035429,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
32558,1.423610,-0.358777,-0.420060,-0.145920,-0.21666,-0.035429,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
32559,-1.215643,0.110960,-0.420060,-0.145920,-0.21666,-1.655225,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [ ]:
features = X_train.columns.to_numpy()

np.save("../data/features.npy", features)

In [ ]:
y_train = y_train.apply(lambda x: 1 if ">50K" in x else 0)
y_test  = y_test.apply(lambda x: 1 if ">50K" in x else 0)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(
    X_train, y_train,
    epochs=5,              # Increase if needed
    batch_size=64,
    validation_split=0.1,  # Splits a portion of the training data for validation
    verbose=1
)

E0000 00:00:1761613198.866655    1146 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1761613198.883914    1146 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/5
458/458 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8420 - loss: 0.3370 - val_accuracy: 0.8511 - val_loss: 0.3156
Epoch 2/5
458/458 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8556 - loss: 0.3088 - val_accuracy: 0.8532 - val_loss: 0.3257
Epoch 3/5
458/458 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8599 - loss: 0.3022 - val_accuracy: 0.8542 - val_loss: 0.3151
Epoch 4/5
458/458 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8611 - loss: 0.3000 - val_accuracy: 0.8551 - val_loss: 0.3133
Epoch 5/5
458/458 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8634 - loss: 0.2949 - val_accuracy: 0.8526 - val_loss: 0.3247


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.8519


In [ ]:
model.save("100x8:8573.h5")